In [2]:
from pathlib import Path
csv_files = list(Path('masking_data').glob("blb_*tsv"))
len(csv_files)

6

In [6]:
import pandas as pd
titles = []
for csv_file in csv_files:
    titles.extend(pd.read_csv(csv_file, sep='\t', index_col=0)['title'].unique())

In [7]:
titles = list(set(titles))
len(titles)

26083

In [8]:
# Load model directly
import torch

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

tokenizer = AutoTokenizer.from_pretrained("TheBritishLibrary/bl-books-genre")
model = AutoModelForSequenceClassification.from_pretrained("TheBritishLibrary/bl-books-genre")
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer,
                      device=device , max_length=256, truncation=True)


config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/332 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [10]:
from tqdm import tqdm
results = []
batch_size = 32
for i in tqdm(range(0, len(titles), batch_size)):
    batch = titles[i:i+batch_size]
    batch_results = classifier(batch)
    results.extend(batch_results)


100%|██████████| 816/816 [06:08<00:00,  2.21it/s]


In [11]:
len(results) == len(titles)

True

In [13]:
df_titles = pd.DataFrame({'title': titles, 'genre': [res['label'] for res in results]})
df_titles.head()

,title,genre
0,"History of the Present Deanery of Bicester, Oxon",Non-fiction
1,Journey across the Western interior of Austral...,Non-fiction
2,Hearts that are Lightest [A tale.],Fiction
3,The Black Sea pilot,Fiction
4,"A Tour in Holland, the Countries on the Rhine,...",Non-fiction


In [ ]:
for csv_file in csv_files:
    df = pd.read_csv(csv_file, sep='\t', index_col=0)
    df = df.merge(df_titles, on='title', how='left')
    df.to_csv(Path('masking_data') / (csv_file.stem + '_with_genres.tsv'), sep='\t')